# **04 - LSTM**

This notebook trains a sequence model on the full frame-by-frame blocker-rusher tracking data, rather than the sampled snapshots used in `03_xgboost.ipynb`, to test whether the trajectory of a block carries predictive information beyond its start and end points. Tracking data is rebuilt and paired in memory as before, then reduced to per-frame geometry features and packed into padded sequences using PyTorch's `pack_padded_sequence`, which lets the LSTM ignore padded frames exactly rather than relying on a masking heuristic. Class imbalance was handled with the same recall-biased weighting used for XGBoost, and training duration was validated directly by comparing freshly-initialized runs at 20 epochs. Permutation importance was computed as the sequence-model equivalent of the XGBoost SHAP analysis, and the trained model and test-set predictions were saved for the model comparison in `05_model_comparison.ipynb`.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, classification_report

snapshot_features = pd.read_parquet("../processed/snapshot_features.parquet")
device = torch.device("mps")

In [ ]:
%run 01_ingest.ipynb
blockers = tracking_df[tracking_df["pff_role"] == "Pass Block"].copy()
rusher_cols = ["gameId", "playId", "frameId", "nflId", "x", "y", "s", "a", "o", "dir"]
rushers = tracking_df[rusher_cols].rename(columns = {
    "nflId": "rusher_nflId",
    "x": "rusher_x", "y": "rusher_y",
    "s": "rusher_s", "a": "rusher_a",
    "o": "rusher_o", "dir": "rusher_dir",
})
paired = blockers.merge(
    rushers,
    left_on = ["gameId", "playId", "frameId", "pff_nflIdBlockedPlayer"],
    right_on = ["gameId", "playId", "frameId", "rusher_nflId"],
    how = "inner",
)
print(f"Paired blocker-rusher frame rows: {len(paired):,}")

In [ ]:
paired["distance"] = np.sqrt(
    (paired["x"] - paired["rusher_x"])**2 + (paired["y"] - paired["rusher_y"])**2
)
paired = paired.sort_values(["gameId", "playId", "nflId", "frameId"])
paired["prev_distance"] = paired.groupby(["gameId", "playId", "nflId"])["distance"].shift(1)
paired["closing_speed"] = (paired["prev_distance"] - paired["distance"]).fillna(0)
paired["orientation_diff"] = (paired["o"] - paired["rusher_o"]).abs()
paired["dx"] = paired["x"] - paired["rusher_x"]
paired["dy"] = paired["y"] - paired["rusher_y"]
paired["speed_diff"] = paired["s"] - paired["rusher_s"]

In [ ]:
seq_feature_cols = ["dx", "dy", "distance", "closing_speed", "orientation_diff", "speed_diff", "s", "a"]
sequences = []
keys = []

for key, group in paired.groupby(["gameId", "playId", "nflId", "rusher_nflId"]):
    group = group.sort_values("frameId")
    sequences.append(group[seq_feature_cols].values.astype("float32"))
    keys.append(key)

seq_lengths = np.array([len(s) for s in sequences])
print(f"Sequences: {len(sequences):,}")
print(f"Sequence length - min: {seq_lengths.min()}, max: {seq_lengths.max()}, median: {int(np.median(seq_lengths))}")

In [ ]:
max_len = int(np.percentile(seq_lengths, 95))
n_features = len(seq_feature_cols)

def pad_sequence(seq, max_len):
    length = min(len(seq), max_len)
    padded = np.zeros((max_len, n_features), dtype = "float32")
    padded[:length] = seq[:length]
    return padded, length

padded_seqs = []
true_lengths = []

for seq in sequences:
    padded, length = pad_sequence(seq, max_len)
    padded_seqs.append(padded)
    true_lengths.append(length)

X_seq = np.stack(padded_seqs)
true_lengths = np.array(true_lengths)
keys_df = pd.DataFrame(keys, columns = ["gameId", "playId", "nflId", "rusher_nflId"])
keys_df = keys_df.merge(
    snapshot_features[["gameId", "playId", "nflId", "rusher_nflId", "blocker_lost"]],
    on = ["gameId", "playId", "nflId", "rusher_nflId"],
    how = "left",
)
y_seq = keys_df["blocker_lost"].values.astype("float32")

print(f"X_seq shape: {X_seq.shape}")
print(f"y_seq shape: {y_seq.shape}, positive rate: {y_seq.mean():.3f}")

In [ ]:
X_train_seq, X_test_seq, y_train_seq, y_test_seq, len_train, len_test, keys_train, keys_test = train_test_split(
    X_seq, y_seq, true_lengths, keys_df, test_size = 0.2, random_state = 42, stratify = y_seq
)
keys_test = keys_test.reset_index(drop=True)
print(f"Train: {X_train_seq.shape}, Test: {X_test_seq.shape}")

In [ ]:
class MatchupDataset(Dataset):
    def __init__(self, X, y, lengths):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32)
        self.lengths = torch.tensor(lengths, dtype = torch.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.lengths[idx]

train_ds = MatchupDataset(X_train_seq, y_train_seq, len_train)
test_ds = MatchupDataset(X_test_seq, y_test_seq, len_test)

train_loader = DataLoader(train_ds, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_ds, batch_size = 64, shuffle = False)

In [ ]:
class BlockOutcomeLSTM(nn.Module):
    def __init__(self, n_features, hidden_size = 32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first = True)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first = True, enforce_sorted = False
        )
        _, (hidden, _) = self.lstm(packed)
        out = self.relu(self.fc1(hidden[-1]))
        out = self.fc2(out)
        return out.squeeze(1)

model = BlockOutcomeLSTM(n_features = n_features).to(device)
print(model)

In [ ]:
torch.manual_seed(525)
model = BlockOutcomeLSTM(n_features=n_features).to(device)

pos_weight = torch.tensor([6.5], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 20

for epoch in range(n_epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch, len_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch, len_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{n_epochs} - train loss: {avg_loss:.4f}")

In [ ]:
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch, len_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch, len_batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

y_pred_proba_lstm = np.array(all_probs)
y_pred_lstm = (y_pred_proba_lstm >= 0.5).astype(int)
y_test_seq_arr = np.array(all_labels)

auc_lstm = roc_auc_score(y_test_seq_arr, y_pred_proba_lstm)
ll_lstm = log_loss(y_test_seq_arr, y_pred_proba_lstm)

print(f"LSTM AUC: {auc_lstm:.4f}")
print(f"LSTM Log-loss: {ll_lstm:.4f}")
print(classification_report(y_test_seq_arr, y_pred_lstm))

In [ ]:
def get_predictions(model, X, lengths, device, batch_size = 64):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.tensor(X[i:i+batch_size], dtype = torch.float32).to(device)
            len_batch = torch.tensor(lengths[i:i+batch_size], dtype = torch.int64)
            logits = model(X_batch, len_batch)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
        return np.array(probs)

baseline_probs = get_predictions(model, X_test_seq, len_test, device)
baseline_auc = roc_auc_score(y_test_seq_arr, baseline_probs)
print(f"Baseline test AUC: {baseline_auc:.4f}")

importance_results = []
rng = np.random.default_rng(42)

for feat_idx, feat_name in enumerate(seq_feature_cols):
    X_permuted = X_test_seq.copy()
    #shuffle this feature's values across the batch dimension, frame by frame, 
    # preserving each sequence's own padding structure and other channels
    for t in range(X_permuted.shape[1]):
        rng.shuffle(X_permuted[:, t, feat_idx])

    permuted_probs = get_predictions(model, X_permuted, len_test, device)
    permuted_auc = roc_auc_score(y_test_seq_arr, permuted_probs)

    importance_results.append({
        "feature": feat_name,
        "baseline_auc": baseline_auc,
        "permuted_auc": permuted_auc,
        "auc_drop": baseline_auc - permuted_auc,
    })

importance_df = pd.DataFrame(importance_results).sort_values("auc_drop", ascending = False).reset_index(drop=True)
importance_df

In [ ]:
lstm_results = keys_test[["gameId", "playId", "nflId", "rusher_nflId"]].copy()
lstm_results["blocker_lost_actual"] = y_test_seq_arr
lstm_results["blocker_lost_predicted_proba"] = y_pred_proba_lstm
lstm_results["blocker_win_probability"] = 1 - lstm_results["blocker_lost_predicted_proba"]

torch.save(model.state_dict(), "lstm_sequence.pt")
lstm_results.to_parquet("lstm_predictions.parquet", index=False)
print("Model and predictions saved.")

In [ ]:
import matplotlib.pyplot as plt

importance_df = importance_df.sort_values("auc_drop", ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(importance_df["feature"], importance_df["auc_drop"], color="#c98a2c")
ax.set_xlabel("AUC drop when shuffled")
ax.set_title("LSTM Permutation Importance")
plt.tight_layout()
plt.savefig("../figures/lstm_permutation_importance.png", dpi=150, bbox_inches="tight")

In [ ]:
import json
import matplotlib.pyplot as plt

with open("../dashboard/data/dashboard_data.json") as f:
    blockers = json.load(f)

position_colors = {"C": "#c98a2c", "G": "#4d4b43", "T": "#a6a396"}

fig, ax = plt.subplots(figsize=(8, 7))

for pos in ["C", "G", "T"]:
    subset = [b for b in blockers if b["blockerPosition"] == pos]
    ax.scatter(
        [b["winRate"] for b in subset],
        [b["winRateOverExpected"] for b in subset],
        color=position_colors[pos], s=70, alpha=0.85, label=pos,
    )

# Label only the extremes so the dense middle stays readable
sorted_by_wroe = sorted(blockers, key=lambda b: b["winRateOverExpected"])
extremes = sorted_by_wroe[:5] + sorted_by_wroe[-5:]
for b in extremes:
    ax.annotate(
        b["blockerName"], (b["winRate"], b["winRateOverExpected"]),
        fontsize=8, xytext=(5, 5), textcoords="offset points"
    )

ax.axhline(0, linestyle="--", color="#a6a396", linewidth=1)
ax.set_xlabel("Win Rate")
ax.set_ylabel("Win Rate Over Expected")
ax.set_title("Blocker Win Rate vs. WROE")
ax.legend(title="Position", loc="lower right")
plt.tight_layout()
plt.savefig("../figures/blocker_wroe_scatter.png", dpi=150, bbox_inches="tight")

In [ ]:
import json
import matplotlib.pyplot as plt

with open("../dashboard/data/dashboard_data.json") as f:
    blockers = json.load(f)

guards = sorted(
    [b for b in blockers if b["blockerPosition"] == "G"],
    key=lambda b: b["winRateOverExpected"]
)

wroe_values = [g["winRateOverExpected"] for g in guards]
bitonio_rank = next(i for i, g in enumerate(guards) if g["blockerName"] == "Joel Bitonio")

print(f"Bitonio rank: {len(guards) - bitonio_rank} of {len(guards)} guards (1 = best)")

colors = ["#c98a2c" if g["blockerName"] == "Joel Bitonio" else "#c7c4b8" for g in guards]

fig, ax = plt.subplots(figsize=(7, 6))
ax.bar(range(len(guards)), wroe_values, color=colors, width=0.9)
ax.axhline(0, color="#4d4b43", linewidth=0.8)
ax.set_xticks([])
ax.set_xlabel(f"All qualifying guards, sorted (n={len(guards)})")
ax.set_ylabel("Win Rate Over Expected")
ax.set_title("Joel Bitonio's WROE Among Guards League-Wide")

plt.tight_layout()
plt.savefig("../figures/bitonio_guard_rank.png", dpi=150, bbox_inches="tight")